In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

from src.loaders import  load_vectorstore, load_reranker
from src.retrievers import create_bm25, create_hybrid_retriever
from src.data_utils import load_chunks
from src.config import RERANK_TOP_K

# Initialize everything
chunks = load_chunks()
vectorstore = load_vectorstore()
bm25 = create_bm25(chunks)
hybrid_retriever = create_hybrid_retriever(bm25, vectorstore)
reranker = load_reranker()

Loading weights: 100%|█████████████████████████████████████████████████████████████| 105/105 [00:00<00:00, 6179.26it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
query = "Explain the attention mechanism in the transformer"

candidates=hybrid_retriever.invoke(query)

pairs=[(query,doc.page_content) for doc in candidates]

scores=reranker.predict(pairs) # iterator return not list


ranked_results=sorted(zip(candidates,scores),key=lambda x:x[1],reverse=True)

print(f"Query: {query}\n" + "="*50)
for i, (doc, score) in enumerate(ranked_results[:RERANK_TOP_K]):
    print(f"\nRank {i+1} (Score: {score:.4f}):\n{doc.page_content[:300]}...")

Query: Explain the attention mechanism in the transformer

Rank 1 (Score: 6.4411):
is similar to that of single-head attention with full dimensionality.
3.2.3 Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and val...

Rank 2 (Score: 5.4469):
block, computing hidden representations in parallel for all input and output positions. In these models,
the number of operations required to relate signals from two arbitrary input or output positions grows
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. Thi...

Rank 3 (Score: 4.9687):
of a single sequence in order to compute a representation of the sequence. Self-attention has been
used successfully in a variety of tasks including reading comprehension, abstractive summarization,
textual entailment and learning task-independent sentence re

In [11]:
ranked_results[0][0].page_content

'is similar to that of single-head attention with full dimensionality.\n3.2.3 Applications of Attention in our Model\nThe Transformer uses multi-head attention in three different ways:\n• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,\nand the memory keys and values come from the output of the encoder. This allows every\nposition in the decoder to attend over all positions in the input sequence. This mimics the\ntypical encoder-decoder attention mechanisms in sequence-to-sequence models such as\n[38, 2, 9].\n• The encoder contains self-attention layers. In a self-attention layer all of the keys, values\nand queries come from the same place, in this case, the output of the previous layer in the\nencoder. Each position in the encoder can attend to all positions in the previous layer of the\nencoder.'

In [12]:
ranked_results[0][1]

np.float32(6.441118)